In [1]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt
import plotly.graph_objects as go

In [2]:
cp_scen_nm = "Current-Policy"
ep_scen_nm = "Enhanced-Ambition"

In [3]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100‑yr GWP (no climate–carbon feedbacks)
GWP_AR5 = {
    'CO2':      1,      
    'CH4':     28,      
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,      
    'N2O_AGR':265,
    'N2O_AWB':265,
    'HFC125': 3170,     
    'HFC134a':1300,     
    'HFC143a':4800,     
    'HFC23': 12400,     
    'HFC32':   677,     
    'HFC43':  1650,     
    'HFC227ea':3350,    
    'HFC236fa':8060,    
    'SF6':   23500,     
    'C2F6':  11100,     
    'CF4':    6630,     
}

In [4]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [6]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_sensitivity"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Enhanced-Ambition-Gas-High, Enhanced-Ambition-Gas-Med, Enhanced-Ambition-Gas-Low, Enhanced-Ambition-Oil-High, Enhanced-Ambition-Oil-Med, Enhanced-Ambition-Oil-Low, Enhanced-Ambition, Enhanced-Ambition


In [7]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Enhanced-Ambition-Gas-High',
 'Enhanced-Ambition-Gas-Med',
 'Enhanced-Ambition-Gas-Low',
 'Enhanced-Ambition-Oil-High',
 'Enhanced-Ambition-Oil-Med',
 'Enhanced-Ambition-Oil-Low',
 'Enhanced-Ambition',
 'Enhanced-Ambition']

In [8]:
scenarios = ['Enhanced-Ambition']

In [9]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [10]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'
q = queries[270]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfGHG = pd.concat([dfCO2, dfNonCO2], axis=0)
dfGHG['emiss(MT)'] = dfGHG.apply(convert_to_mt, axis=1)
dfGHG['gwpAr5'] = dfGHG['GHG'].apply(lambda gas: GWP_AR5[gas] if gas in GWP_AR5 else np.nan)
dfGHG['MTCO2eq'] = dfGHG['emiss(MT)'] * dfGHG['gwpAr5']
dfGHG

CO2 emissions by sector (no bio) (excluding resource production)


nonCO2 emissions by sector (excluding resource production)


,Units,scenario,region,sector,Year,value,GHG,emiss(MT),gwpAr5,MTCO2eq
0,MTC,Enhanced-Ambition,South Korea,H2 central production,2020,0.002300,CO2,0.008434,1.0,0.008434
1,MTC,Enhanced-Ambition,South Korea,H2 central production,2025,0.013499,CO2,0.049496,1.0,0.049496
2,MTC,Enhanced-Ambition,South Korea,H2 central production,2030,0.002237,CO2,0.008204,1.0,0.008204
3,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,0.001685,CO2,0.006179,1.0,0.006179
4,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2020,0.002906,CO2,0.010656,1.0,0.010656
...,...,...,...,...,...,...,...,...,...,...
7837,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2015,0.000206,SO2_2,0.000206,NaN,NaN
7838,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2020,0.000209,SO2_2,0.000209,NaN,NaN
7839,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2025,0.000205,SO2_2,0.000205,NaN,NaN
7840,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2030,0.000203,SO2_2,0.000203,NaN,NaN


In [11]:
dfGHG['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'airCO2', 'ammonia',
       'backup_electricity', 'cement', 'chemical energy use',
       'chemical feedstocks', 'comm cooling', 'comm heating',
       'comm others', 'construction energy use',
       'construction feedstocks', 'delivered biomass', 'delivered gas',
       'desalinated water', 'elec_coal (IGCC CCS)',
       'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)', 'electricity',
       'gas pipeline', 'gas processing', 'iron and steel',
       'mining energy use', 'other industrial energy use',
       'other industrial feedstocks', 'process heat cement',
       'process heat dac', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal_d1',
       'resid heating coal_d10', 'resid heating coal_d2',
       'resid heating coal_d3', 'resid heating coal_d4',
       'resid heating coal_d5', 'res

In [12]:
dfOut = dfGHG[~(dfGHG['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum().reset_index()
dfOut.rename(columns={'MTCO2eq': 'value'}, inplace=True)
dfOut['Units'] = 'MTCO2eq'

In [13]:
dfOut = dfOut.set_index(['scenario', 'Year'])
dfOut.loc[(ep_scen_nm, 2005), 'value'] -= 57.5
dfOut.loc[(ep_scen_nm, 2010), 'value'] -= 57.3
dfOut.loc[(ep_scen_nm, 2015), 'value'] -= 47.8
dfOut.loc[(ep_scen_nm, 2020), 'value'] -= 38.8
dfOut.loc[(ep_scen_nm, 2025), 'value'] -= 30.4
dfOut.loc[(ep_scen_nm, 2030), 'value'] -= 38.8
dfOut.loc[(ep_scen_nm, 2035), 'value'] -= 47.8
dfOut = dfOut.reset_index()

In [14]:
dfOut

,scenario,Year,value,Units
0,Enhanced-Ambition,1975,53.918215,MTCO2eq
1,Enhanced-Ambition,1990,291.111739,MTCO2eq
2,Enhanced-Ambition,2005,510.346007,MTCO2eq
3,Enhanced-Ambition,2010,612.386814,MTCO2eq
4,Enhanced-Ambition,2015,671.402402,MTCO2eq
5,Enhanced-Ambition,2020,673.131363,MTCO2eq
6,Enhanced-Ambition,2025,644.613378,MTCO2eq
7,Enhanced-Ambition,2030,465.332480,MTCO2eq
8,Enhanced-Ambition,2035,310.550852,MTCO2eq


In [15]:
783.8 * 0.6

470.28

In [18]:
1 - 310.55 / 783.8

0.6037892319469251

In [17]:
dfPower = dfGHG[(dfGHG['sector'].str.contains('elec'))]# | (dfGHG['sector'].isin(lstOtherPower))].copy()
dfOutPower = dfPower.groupby(['scenario', 'region', 'Year'])['MTCO2eq'].sum().reset_index()
dfOutPower.rename(columns={'MTCO2eq': 'value'}, inplace=True)
dfOutPower['Units'] = 'MTCO2eq'
dfOutPower

,scenario,region,Year,value,Units
0,Enhanced-Ambition,South Korea,1990,35.315064,MTCO2eq
1,Enhanced-Ambition,South Korea,2005,177.820212,MTCO2eq
2,Enhanced-Ambition,South Korea,2010,251.589396,MTCO2eq
3,Enhanced-Ambition,South Korea,2015,260.227190,MTCO2eq
4,Enhanced-Ambition,South Korea,2020,249.307269,MTCO2eq
5,Enhanced-Ambition,South Korea,2025,212.391710,MTCO2eq
6,Enhanced-Ambition,South Korea,2030,144.097166,MTCO2eq
7,Enhanced-Ambition,South Korea,2035,63.550737,MTCO2eq


In [40]:
fig = go.Figure()

# emiss_2018 = 727.6
emiss_2018 = 783.8
# emiss_2018_net = 783.81
x_center = 2040
width = 1
# offset = 4.9574  # BPESD upward shift

emiss_2035_cur = dfOut.set_index(['scenario', 'Year'])['value'].get((cp_scen_nm, 2035))
# emiss_2035_enh = dfOut.set_index(['scenario', 'Year'])['value'].get((ep_scen_nm, 2035))

reduc_cur = (1 - (emiss_2035_cur / emiss_2018)) * 100
# reduc_enh = (1 - (emiss_2035_enh / emiss_2018)) * 100

# === Legend Group: Gases ===
# fig.add_trace(go.Scatter(
#     x=[None], y=[None], mode='lines',
#     line=dict(color='rgba(0,0,0,0)'),
#     name='<b>Sector</b>', showlegend=True, hoverinfo='skip'
# ))

# fig.add_trace(go.Scatter(
#     x=dfHist['index'],
#     y=dfHist['순배출량'] / 1000,
#     mode='lines', name='Total GHG (incl. LULUCF)',
#     line=dict(color='black', width=1)
# ))


# fig.add_trace(go.Scatter(
#     x=dfHist['index'],
#     y=dfHist['a. 공공전기 및 열 생산'] / 1000,
#     mode='lines', name='Electricity',
#     line=dict(color='black', width=1, dash='dash')
# ))

# === Legend Group: Gases ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Gas</b>', showlegend=True, hoverinfo='skip'
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CO2_stack'],
    mode='lines', name='CO2', fill='tozeroy',
    line=dict(width=0.5, color='grey')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CH4_stack'],
    mode='lines', name='CH4', fill='tonexty',
    line=dict(width=0.5, color='green')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['N2O_stack'],
    mode='lines', name='N2O', fill='tonexty',
    line=dict(width=0.5, color='purple')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['FGas_stack'],
    mode='lines', name='F-Gases', fill='tonexty',
    line=dict(width=0.5, color='yellow')
))

fig.add_trace(go.Scatter(
    x=[2030], y=[emiss_2018 * 0.6],
    mode='markers+text',
    marker=dict(color='#750D86', size=7, symbol='triangle-up'),
    text='2030 NDC', textposition="middle left",
    textfont=dict(size=15, color="#750D86"),
    showlegend=False
))

# === Static Reference Point ===
fig.add_trace(go.Scatter(
    x=[2018], y=[emiss_2018],
    mode='markers+text',
    marker=dict(color='black', size=7),
    text=f'2018: {emiss_2018} MtCO2e',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))

# === Legend Group: Scenarios ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Scenario</b>', showlegend=True, hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=dfHist['index'],
    y=dfHist['순배출량'] / 1000,
    mode='lines', name='Historical Data',
    line=dict(color='black', width=1)
))

fig.add_trace(go.Scatter(
    x=dfOut[(dfOut['scenario'] == cp_scen_nm) & (dfOut['Year'] >= 2025)]['Year'],
    y=dfOut[(dfOut['scenario'] == cp_scen_nm) & (dfOut['Year'] >= 2025)]['value'],
    mode='lines', name='Current Policy',
    line=dict(color='#636EFA', width=1.5)
))

# fig.add_trace(go.Scatter(
#     x=dfOut[(dfOut['scenario'] == ep_scen_nm) & (dfOut['Year'] >= 2025)]['Year'],
#     y=dfOut[(dfOut['scenario'] == ep_scen_nm) & (dfOut['Year'] >= 2025)]['value'],
#     mode='lines', name='Enhanced Ambition',
#     line=dict(color='#00CC96', width=1.5)
# ))

# fig.add_trace(go.Scatter(
#     x=dfOutPower[(dfOutPower['scenario'] == cp_scen_nm) & (dfOutPower['Year'] >= 2025)]['Year'],
#     y=dfOutPower[(dfOutPower['scenario'] == cp_scen_nm) & (dfOutPower['Year'] >= 2025)]['value'],
#     mode='lines', name='Current Policy',
#     line=dict(color='#636EFA', width=1.5, dash='dash'),
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(
#     x=dfOutPower[(dfOutPower['scenario'] == ep_scen_nm) & (dfOutPower['Year'] >= 2005)]['Year'],
#     y=dfOutPower[(dfOutPower['scenario'] == ep_scen_nm) & (dfOutPower['Year'] >= 2005)]['value'],
#     mode='lines', name='Enhanced Ambition',
#     line=dict(color='#00CC96', width=1.5, dash='dash'),
#     showlegend=False
# ))

# === Axes & Layout ===
fig.update_layout(
    # title="<b>Greenhouse gas emission pathways of South Korea</b>",
    # title_font_size=28,
    # title_x=0.43,
    legend_traceorder="normal",
    legend_title='',
    legend_font_size=15,
    plot_bgcolor='rgba(0,0,0,0)',
    width=1200,
    height=700,
    xaxis=dict(
        showgrid=False,
        title="Year", title_font_size=25,
        tickvals=list(range(1990, 2040, 5)),
        tickfont_size=15,
        range=[1987, 2045]
    ),
    yaxis=dict(
        showgrid=False,
        title="Emission (MtCO2e)", title_font_size=25,
        tickvals=list(range(0, 801, 100)),
        tickfont_size=15,
        range=[-30, 830]
    )
)

# === Reduction Boxes (overlay bars) ===
# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[emiss_2035_cur, emiss_2035_cur, emiss_2035_enh, emiss_2035_enh],
#     fill='toself', fillcolor='rgba(0,204,150,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))
fig.add_trace(go.Scatter(
    x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
    y=[emiss_2018, emiss_2018, emiss_2035_cur, emiss_2035_cur],
    fill='toself', fillcolor='rgba(99,110,250,0.3)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
))

# === Reduction Labels ===
fig.add_annotation(
    x=x_center - 0.15, y=(emiss_2018 + emiss_2035_cur) / 2,
    text=f"<b>Current<br>Policy<br>-{reduc_cur:.1f}%</b>",
    showarrow=False, font=dict(size=14, color="#636EFA")
)
# fig.add_annotation(
#     x=x_center + 0.15, y=(emiss_2035_enh + emiss_2035_cur) / 2,
#     text=f"<b>Enhanced<br>Ambition<br>-{reduc_enh:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#00CC96")
# )

# === Reduction Boxes (overlay bars) ===
# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[119.0679, 119.0679, 40.657, 40.657],
#     fill='toself', fillcolor='rgba(0,204,150,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))

# fig.add_trace(go.Bar(
#     x=[x_center],
#     y=[278.84-40.657],
#     base=[40.657],  # for positioning
#     width=width,
#     marker=dict(
#         color='rgba(99,110,250,0.3)',
#         pattern=dict(
#             shape='\\',  # options: '/', '\\', 'x', '-', '|', '+', '.'
#             fillmode='overlay',
#             size=5,
#             solidity=0.2
#         )
#     ),
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[278.84, 278.84, 119.0679, 119.0679],
#     fill='toself', fillcolor='rgba(99,110,250,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))

# reduc_cur_power = 100 * (1 - 119.0679/278.84)
# reduc_enh_power = 100 * (1 - 40.657/278.84)

# # === Reduction Labels ===
# fig.add_annotation(
#     x=x_center - 0.15, y=(278.84 + 119.0679) / 2,
#     text=f"<b>-{reduc_cur_power:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#636EFA")
# )
# fig.add_annotation(
#     x=x_center + 0.15, y=(40.657 + 119.0679) / 2,
#     text=f"<b>-{reduc_enh_power:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#00CC96")
# )


# === Grid Lines (manual dashed lines) ===
for x in range(1990, 2040, 5):
    fig.add_shape(type="line", x0=x, x1=x, y0=-30, y1=830,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
for y in range(0, 831, 100):
    fig.add_shape(type="line", x0=1987, x1=2037, y0=y, y1=y,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
fig.add_shape(type="line", x0=1987, x1=2037, y0=0, y1=0,
              line=dict(color="LightGrey", width=1, dash="dash"),
              layer='above')

# === Optional: Highlight window 2025–2035 ===
fig.add_vrect(x0=2025, x1=2035, fillcolor="LightBlue", opacity=0.1,
              layer="below", line_width=0)

# === Box Around Plot ===
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=0, x1=1, y0=0, y1=1,
    line=dict(color="black", width=1),
    layer="above"
)

fig.add_annotation(
    text="w/o International Offset",
    xref="paper", 
    yref="paper",
    x=0.5,
    y=1.08,
    showarrow=False,
    font=dict(size=21, color="grey"),
    align="center"
)

fig

In [41]:
fig = go.Figure()

# emiss_2018 = 727.6
emiss_2018 = 783.8
# emiss_2018_net = 783.81
x_center = 2040
width = 1
# offset = 4.9574  # BPESD upward shift

emiss_2035_cur = dfOut.set_index(['scenario', 'Year'])['value'].get((cp_scen_nm, 2035))
emiss_2035_enh = dfOut.set_index(['scenario', 'Year'])['value'].get((ep_scen_nm, 2035))

reduc_cur = (1 - (emiss_2035_cur / emiss_2018)) * 100
reduc_enh = (1 - (emiss_2035_enh / emiss_2018)) * 100

# === Legend Group: Gases ===
# fig.add_trace(go.Scatter(
#     x=[None], y=[None], mode='lines',
#     line=dict(color='rgba(0,0,0,0)'),
#     name='<b>Sector</b>', showlegend=True, hoverinfo='skip'
# ))

# fig.add_trace(go.Scatter(
#     x=dfHist['index'],
#     y=dfHist['순배출량'] / 1000,
#     mode='lines', name='Total GHG (incl. LULUCF)',
#     line=dict(color='black', width=1)
# ))


# fig.add_trace(go.Scatter(
#     x=dfHist['index'],
#     y=dfHist['a. 공공전기 및 열 생산'] / 1000,
#     mode='lines', name='Electricity',
#     line=dict(color='black', width=1, dash='dash')
# ))

# === Legend Group: Gases ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Gas</b>', showlegend=True, hoverinfo='skip'
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CO2_stack'],
    mode='lines', name='CO2', fill='tozeroy',
    line=dict(width=0.5, color='grey')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CH4_stack'],
    mode='lines', name='CH4', fill='tonexty',
    line=dict(width=0.5, color='green')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['N2O_stack'],
    mode='lines', name='N2O', fill='tonexty',
    line=dict(width=0.5, color='purple')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['FGas_stack'],
    mode='lines', name='F-Gases', fill='tonexty',
    line=dict(width=0.5, color='yellow')
))

fig.add_trace(go.Scatter(
    x=[2030], y=[emiss_2018 * 0.6],
    mode='markers+text',
    marker=dict(color='#750D86', size=7, symbol='triangle-up'),
    text='2030 NDC', textposition="middle left",
    textfont=dict(size=15, color="#750D86"),
    showlegend=False
))

# === Static Reference Point ===
fig.add_trace(go.Scatter(
    x=[2018], y=[emiss_2018],
    mode='markers+text',
    marker=dict(color='black', size=7),
    text=f'2018: {emiss_2018} MtCO2e',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))

# === Legend Group: Scenarios ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Scenario</b>', showlegend=True, hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=dfHist['index'],
    y=dfHist['순배출량'] / 1000,
    mode='lines', name='Historical Data',
    line=dict(color='black', width=1)
))

fig.add_trace(go.Scatter(
    x=dfOut[(dfOut['scenario'] == cp_scen_nm) & (dfOut['Year'] >= 2025)]['Year'],
    y=dfOut[(dfOut['scenario'] == cp_scen_nm) & (dfOut['Year'] >= 2025)]['value'],
    mode='lines', name='Current Policy',
    line=dict(color='#636EFA', width=1.5)
))

fig.add_trace(go.Scatter(
    x=dfOut[(dfOut['scenario'] == ep_scen_nm) & (dfOut['Year'] >= 2025)]['Year'],
    y=dfOut[(dfOut['scenario'] == ep_scen_nm) & (dfOut['Year'] >= 2025)]['value'],
    mode='lines', name='Enhanced Ambition',
    line=dict(color='#00CC96', width=1.5)
))

# fig.add_trace(go.Scatter(
#     x=dfOutPower[(dfOutPower['scenario'] == cp_scen_nm) & (dfOutPower['Year'] >= 2025)]['Year'],
#     y=dfOutPower[(dfOutPower['scenario'] == cp_scen_nm) & (dfOutPower['Year'] >= 2025)]['value'],
#     mode='lines', name='Current Policy',
#     line=dict(color='#636EFA', width=1.5, dash='dash'),
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(
#     x=dfOutPower[(dfOutPower['scenario'] == ep_scen_nm) & (dfOutPower['Year'] >= 2005)]['Year'],
#     y=dfOutPower[(dfOutPower['scenario'] == ep_scen_nm) & (dfOutPower['Year'] >= 2005)]['value'],
#     mode='lines', name='Enhanced Ambition',
#     line=dict(color='#00CC96', width=1.5, dash='dash'),
#     showlegend=False
# ))

# === Axes & Layout ===
fig.update_layout(
    # title="<b>Greenhouse gas emission pathways of South Korea</b>",
    # title_font_size=28,
    # title_x=0.43,
    legend_traceorder="normal",
    legend_title='',
    legend_font_size=15,
    plot_bgcolor='rgba(0,0,0,0)',
    width=1200,
    height=700,
    xaxis=dict(
        showgrid=False,
        title="Year", title_font_size=25,
        tickvals=list(range(1990, 2040, 5)),
        tickfont_size=15,
        range=[1987, 2045]
    ),
    yaxis=dict(
        showgrid=False,
        title="Emission (MtCO2e)", title_font_size=25,
        tickvals=list(range(0, 801, 100)),
        tickfont_size=15,
        range=[-30, 830]
    )
)

# === Reduction Boxes (overlay bars) ===
fig.add_trace(go.Scatter(
    x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
    y=[emiss_2035_cur, emiss_2035_cur, emiss_2035_enh, emiss_2035_enh],
    fill='toself', fillcolor='rgba(0,204,150,0.3)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
))
fig.add_trace(go.Scatter(
    x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
    y=[emiss_2018, emiss_2018, emiss_2035_cur, emiss_2035_cur],
    fill='toself', fillcolor='rgba(99,110,250,0.3)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
))

# === Reduction Labels ===
fig.add_annotation(
    x=x_center - 0.15, y=(emiss_2018 + emiss_2035_cur) / 2,
    text=f"<b>Current<br>Policy<br>-{reduc_cur:.1f}%</b>",
    showarrow=False, font=dict(size=14, color="#636EFA")
)
fig.add_annotation(
    x=x_center + 0.15, y=(emiss_2035_enh + emiss_2035_cur) / 2,
    text=f"<b>Enhanced<br>Ambition<br>-{reduc_enh:.1f}%</b>",
    showarrow=False, font=dict(size=14, color="#00CC96")
)

# === Reduction Boxes (overlay bars) ===
# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[119.0679, 119.0679, 40.657, 40.657],
#     fill='toself', fillcolor='rgba(0,204,150,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))

# fig.add_trace(go.Bar(
#     x=[x_center],
#     y=[278.84-40.657],
#     base=[40.657],  # for positioning
#     width=width,
#     marker=dict(
#         color='rgba(99,110,250,0.3)',
#         pattern=dict(
#             shape='\\',  # options: '/', '\\', 'x', '-', '|', '+', '.'
#             fillmode='overlay',
#             size=5,
#             solidity=0.2
#         )
#     ),
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[278.84, 278.84, 119.0679, 119.0679],
#     fill='toself', fillcolor='rgba(99,110,250,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))

# reduc_cur_power = 100 * (1 - 119.0679/278.84)
# reduc_enh_power = 100 * (1 - 40.657/278.84)

# # === Reduction Labels ===
# fig.add_annotation(
#     x=x_center - 0.15, y=(278.84 + 119.0679) / 2,
#     text=f"<b>-{reduc_cur_power:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#636EFA")
# )
# fig.add_annotation(
#     x=x_center + 0.15, y=(40.657 + 119.0679) / 2,
#     text=f"<b>-{reduc_enh_power:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#00CC96")
# )


# === Grid Lines (manual dashed lines) ===
for x in range(1990, 2040, 5):
    fig.add_shape(type="line", x0=x, x1=x, y0=-30, y1=830,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
for y in range(0, 831, 100):
    fig.add_shape(type="line", x0=1987, x1=2037, y0=y, y1=y,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
fig.add_shape(type="line", x0=1987, x1=2037, y0=0, y1=0,
              line=dict(color="LightGrey", width=1, dash="dash"),
              layer='above')

# === Optional: Highlight window 2025–2035 ===
fig.add_vrect(x0=2025, x1=2035, fillcolor="LightBlue", opacity=0.1,
              layer="below", line_width=0)

# === Box Around Plot ===
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=0, x1=1, y0=0, y1=1,
    line=dict(color="black", width=1),
    layer="above"
)

fig.add_annotation(
    text="w/o International Offset",
    xref="paper", 
    yref="paper",
    x=0.5,
    y=1.08,
    showarrow=False,
    font=dict(size=21, color="grey"),
    align="center"
)

fig

In [42]:
fig = go.Figure()

# emiss_2018 = 727.6
emiss_2018 = 783.8

# === Legend Group: Gases ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Gas</b>', showlegend=True, hoverinfo='skip'
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CO2_stack'],
    mode='lines', name='CO2', fill='tozeroy',
    line=dict(width=0.5, color='grey')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CH4_stack'],
    mode='lines', name='CH4', fill='tonexty',
    line=dict(width=0.5, color='green')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['N2O_stack'],
    mode='lines', name='N2O', fill='tonexty',
    line=dict(width=0.5, color='purple')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['FGas_stack'],
    mode='lines', name='F-Gases', fill='tonexty',
    line=dict(width=0.5, color='yellow')
))

fig.add_trace(go.Scatter(
    x=[2030], y=[emiss_2018 * 0.6],
    mode='markers+text',
    marker=dict(color='#750D86', size=7, symbol='triangle-up'),
    text='2030 NDC', textposition="middle left",
    textfont=dict(size=15, color="#750D86"),
    showlegend=False
))


# === Static Reference Point ===
fig.add_trace(go.Scatter(
    x=[2018], y=[emiss_2018],
    mode='markers+text',
    marker=dict(color='black', size=7),
    text=f'2018: {emiss_2018} MtCO2e',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))
# === Legend Group: Scenarios ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Scenario</b>', showlegend=True, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=dfHist['index'],
    y=dfHist['순배출량'] / 1000,
    mode='lines', name='Historical Data',
    line=dict(color='black', width=1)
))

# === Axes & Layout ===
fig.update_layout(
    # title="<b>Historical Greenhouse gas emissions of Korea</b>",
    # title_font_size=28,
    # title_x=0.43,
    legend_traceorder="normal",
    legend_title='',
    legend_font_size=15,
    plot_bgcolor='rgba(0,0,0,0)',
    width=1200,
    height=700,
    xaxis=dict(
        showgrid=False,
        title="Year", title_font_size=25,
        tickvals=list(range(1990, 2055, 5)),
        tickfont_size=15,
        range=[1987, 2055]
    ),
    yaxis=dict(
        showgrid=False,
        title="Emission (MtCO2e)", title_font_size=25,
        tickvals=list(range(0, 801, 100)),
        tickfont_size=15,
        range=[-30, 830]
    )
)


# === Grid Lines (manual dashed lines) ===
for x in range(1990, 2055, 5):
    fig.add_shape(type="line", x0=x, x1=x, y0=-30, y1=830,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
for y in range(0, 831, 100):
    fig.add_shape(type="line", x0=1987, x1=2055, y0=y, y1=y,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
fig.add_shape(type="line", x0=1987, x1=2055, y0=0, y1=0,
              line=dict(color="LightGrey", width=1, dash="dash"),
              layer='above')

# === Optional: Highlight window 2025–2035 ===
fig.add_vrect(x0=2025, x1=2055, fillcolor="LightBlue", opacity=0.1,
              layer="below", line_width=0)

# === Box Around Plot ===
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=0, x1=1, y0=0, y1=1,
    line=dict(color="black", width=1),
    layer="above"
)

fig.add_trace(go.Scatter(
    x=[2018, 2050],
    y=[783.8,   0],
    mode="lines",
    line=dict(color="black", dash="dash", width=2),
    name="2050 Net-Zero"
))


# fig.add_annotation(
#     text="w/o International Offset",
#     xref="paper", 
#     yref="paper",
#     x=0.5,
#     y=1.08,
#     showarrow=False,
#     font=dict(size=21, color="grey"),
#     align="center"
# )

# 1) 기울기 계산
slope = (783.8 - 310.57) / (2018 - 1990)  # ≈ 16.81 MtCO2e per year

# 2) 1990→2018 dashed 선 추가
fig.add_shape(
    type="line",
    xref="x", yref="y",
    x0=1990, y0=310.57,
    x1=2018, y1=783.8,
    line=dict(color="black", dash="dash", width=2),
    layer="above"
)

# 3) 중간 지점에 화살표와 함께 기울기 표시
mid_x = 2004
mid_y = (310.57 + 783.8) / 2

fig.add_annotation(
    x=mid_x, y=mid_y,
    text=f"Slope ≈ {slope:.2f} MtCO₂e/yr",
    showarrow=True,
    arrowhead=2,
    ax=-80,    # 화살표 꼬리 위치 조정
    ay=-40,
    font=dict(size=14, color="black")
)

# 1) Calculate net-zero slope from 2018→2050
slope_nz = (0 - 783.8) / (2050 - 2018)  # ≈ -24.49 MtCO₂e/yr

# 3) Annotate the slope at the midpoint
mid_x = (2018 + 2050) / 2       # 2034
mid_y = (783.8 +   0) / 2       # 391.9

fig.add_annotation(
    x=mid_x, y=mid_y,
    text=f"Slope ≈ {slope_nz:.2f} MtCO₂e/yr",
    showarrow=True,
    arrowhead=2,
    ax=70,    # tweak horizontal offset of the arrow tail
    ay=-90,   # tweak vertical offset
    font=dict(size=14, color="black")
)

fig